# 01 · Extracción y limpieza — API CMF Bancos

**Objetivo:** obtener los datos del sistema bancario chileno desde la API oficial de la CMF,
limpiarlos y exportarlos a `data/clean/` para alimentar el dashboard de Power BI.

**Fuente:** [api.cmfchile.cl](https://api.cmfchile.cl/) — acceso gratuito con API Key.

---

## 0 · Configuración

In [ ]:
import os
import json
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

API_KEY  = os.getenv('CMF_API_KEY')
BASE_URL = 'https://api.cmfchile.cl/api-sbifv3/recursos_api'

RAW_DIR   = Path('..') / 'data' / 'raw'
CLEAN_DIR = Path('..') / 'data' / 'clean'
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print('API Key cargada:', bool(API_KEY))

## 1 · Funciones auxiliares

In [ ]:
def api_get(endpoint: str, params: dict | None = None) -> dict:
    """Llama a la API CMF y devuelve el JSON."""
    url = f'{BASE_URL}/{endpoint}'
    p = {'apikey': API_KEY, 'formato': 'json', **(params or {})}
    r = requests.get(url, params=p, timeout=30)
    r.raise_for_status()
    return r.json()


def limpiar_numeros(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Convierte columnas con formato numérico chileno a float.
    Decisión: la CMF devuelve los montos como strings con puntos de miles y
    coma decimal (ej. '1.234.567,89'). Se eliminan los puntos y se reemplaza
    la coma por punto antes de convertir. Los valores vacíos o no numéricos
    se convierten a NaN (pandas los ignorará en los cálculos de Power BI).
    """
    for col in cols:
        if col not in df.columns:
            continue
        df[col] = (
            df[col].astype(str)
            .str.strip()
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
            .pipe(pd.to_numeric, errors='coerce')
        )
    return df


def guardar_csv(df: pd.DataFrame, nombre: str) -> None:
    """Guarda en data/clean/ como CSV UTF-8 con BOM (compatible con Excel/Power BI)."""
    ruta = CLEAN_DIR / f'{nombre}.csv'
    df.to_csv(ruta, index=False, encoding='utf-8-sig')
    print(f'  → {ruta.name} ({len(df)} filas, {len(df.columns)} columnas)')

## 2 · Extracción: colocaciones

In [ ]:
# Período a consultar (formato AAAAMM). Ajustar según disponibilidad de la API.
PERIODOS = ['202212', '202212', '202312', '202406']

frames_col = []

for periodo in PERIODOS:
    print(f'Consultando colocaciones {periodo}...')
    data = api_get(f'colocaciones/{periodo}')
    # Guardamos el raw para reproducibilidad
    (RAW_DIR / f'colocaciones_{periodo}.json').write_text(
        json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    df = pd.DataFrame(data.get('Colocaciones', []))
    df['periodo'] = periodo
    frames_col.append(df)

df_col = pd.concat(frames_col, ignore_index=True)
print(f'Total registros: {len(df_col)}')
df_col.head()

## 3 · Limpieza: colocaciones

Decisiones de limpieza:
- Se renombran columnas a nombres descriptivos en español minúscula con guión bajo.
- Las columnas de monto se convierten de string (formato CMF) a float en millones de pesos.
- Se descarta la columna `Codigo` si existe y ya tenemos `RUT`; aporta redundancia sin valor analítico.

In [ ]:
# Mapeo de columnas (ajustar según respuesta real de la API)
RENAME_COL = {
    'Banco': 'banco',
    'RUT': 'rut',
    'Periodo': 'periodo',
    'ColComerciales': 'col_comercial',
    'ColConsumo': 'col_consumo',
    'ColVivienda': 'col_vivienda',
    'Total': 'col_total',
}

df_col = df_col.rename(columns={k: v for k, v in RENAME_COL.items() if k in df_col.columns})

cols_num = ['col_comercial', 'col_consumo', 'col_vivienda', 'col_total']
df_col = limpiar_numeros(df_col, cols_num)

# Columna de período como fecha (primer día del mes)
df_col['fecha'] = pd.to_datetime(df_col['periodo'], format='%Y%m')

print(df_col.dtypes)
df_col.describe()

In [ ]:
guardar_csv(df_col, 'colocaciones')

## 4 · Extracción y limpieza: morosidad / cartera vencida

In [ ]:
frames_mor = []

for periodo in PERIODOS:
    print(f'Consultando morosidad {periodo}...')
    data = api_get(f'cartera_vencida/{periodo}')
    (RAW_DIR / f'morosidad_{periodo}.json').write_text(
        json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    df = pd.DataFrame(data.get('CartVencida', []))
    df['periodo'] = periodo
    frames_mor.append(df)

df_mor = pd.concat(frames_mor, ignore_index=True)
print(f'Total registros: {len(df_mor)}')
df_mor.head()

In [ ]:
RENAME_MOR = {
    'Banco': 'banco',
    'RUT': 'rut',
    'Periodo': 'periodo',
    'CartVencida': 'cartera_vencida',
    'Provisiones': 'provisiones',
    'Total': 'col_total',
}

df_mor = df_mor.rename(columns={k: v for k, v in RENAME_MOR.items() if k in df_mor.columns})

cols_num_mor = ['cartera_vencida', 'provisiones', 'col_total']
df_mor = limpiar_numeros(df_mor, cols_num_mor)

df_mor['fecha'] = pd.to_datetime(df_mor['periodo'], format='%Y%m')

guardar_csv(df_mor, 'morosidad')

## 5 · Verificación de calidad

In [ ]:
print('=== Colocaciones ===')
print(f'Filas: {len(df_col)} | Nulos totales: {df_col.isna().sum().sum()}')
print(df_col[cols_num].describe().round(1))

print('\n=== Morosidad ===')
print(f'Filas: {len(df_mor)} | Nulos totales: {df_mor.isna().sum().sum()}')
print(df_mor[cols_num_mor].describe().round(1))

---
**Próximos pasos:** cargar los CSV de `data/clean/` en Power BI y crear las medidas DAX para ROE, ROA, índice de morosidad y crecimiento real 12 meses.